### Static Neural Architecture

- Batch 1: Digits 0-4
- Batch 2: Digits 5-7
- Batch 3: Digits 8,9

Mechanisms compared:
- Naive fine-tuning
- Rehearsal buffer
- EWC (Elastic Weight Consolidation) with improved regularization
- Synaptic Intelligence (SI) with improved regularization
- Generative replay (stubbed as rehearsal; you can later plug a VAE/GAN)
  
In the static-output setup, the network is built once with 10 output nodes (for all MNIST digits) from the start.
That means when we train on batch 1 (digits 0-4), the output layer already has slots for digits 5-9, but those logits are simply never trained yet. So when batch 2 and 3 arrive, no new neurons are added  the architecture is fixed. Thats why the report said units=0, connections=0.

This is the most common baseline in class-incremental learning (CIL): a fixed classifier head with all classes pre-allocated, and the challenge is to keep the earlier weights from being overwritten. Its simple, but it does not mimic the growing output layer intuition.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# -------------------------------
# Load and preprocess MNIST
# -------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

# Class-incremental batches
batches = [(0, 5), (5, 8), (8, 10)]
train_batches, test_batches = [], []
for start, end in batches:
    idx_tr = np.isin(y_train, np.arange(start, end))
    idx_te = np.isin(y_test, np.arange(start, end))
    train_batches.append((x_train[idx_tr], y_train[idx_tr]))
    test_batches.append((x_test[idx_te], y_test[idx_te]))

# -------------------------------
# Model builder (static 10-way MLP)
# -------------------------------
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    return model

# -------------------------------
# Rehearsal buffer (reservoir)
# -------------------------------
class RehearsalBuffer:
    def __init__(self, size=1000):
        self.size = size
        self.data = None
        self.labels = None

    def add_batch(self, x, y):
        if self.data is None:
            cap = min(len(x), self.size)
            self.data = x[:cap].copy()
            self.labels = y[:cap].copy()
        else:
            for i in range(len(x)):
                if len(self.data) < self.size:
                    self.data = np.vstack([self.data, x[i:i+1]])
                    self.labels = np.hstack([self.labels, y[i:i+1]])
                else:
                    j = np.random.randint(0, self.size)
                    self.data[j] = x[i]
                    self.labels[j] = y[i]

    def get(self):
        return self.data, self.labels

# -------------------------------
# EWC: store numpy snapshots
# -------------------------------
class EWC:
    def __init__(self, model, lam=100.0):
        self.model = model
        self.lam = lam
        self.star_vars = None
        self.fisher = None

    def compute_fisher(self, x, y, sample_size=1000):
        if len(x) == 0:
            return
        idx = np.random.choice(len(x), min(sample_size, len(x)), replace=False)
        x_s = tf.convert_to_tensor(x[idx])
        y_s = tf.convert_to_tensor(y[idx])
        with tf.GradientTape() as tape:
            preds = self.model(x_s, training=False)
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(y_s, preds)
            )
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.fisher = [
            (g.numpy() ** 2) if g is not None else np.zeros_like(v.numpy())
            for g, v in zip(grads, self.model.trainable_variables)
        ]
        self.star_vars = [v.numpy().copy() for v in self.model.trainable_variables]

    def penalty_value(self):
        if self.fisher is None or self.star_vars is None:
            return 0.0
        total = 0.0
        for v, v_star, f in zip(self.model.trainable_variables, self.star_vars, self.fisher):
            total += np.sum(f * np.square(v.numpy() - v_star))
        return (self.lam / 2.0) * total

# -------------------------------
# Synaptic Intelligence (SI)
# -------------------------------
class SITracker:
    def __init__(self, model, c=1.0, eps=1e-3):
        self.model = model
        self.c = c
        self.eps = eps
        self.prev_vars = [v.numpy().copy() for v in self.model.trainable_variables]
        self.w_importance = [np.zeros_like(v.numpy()) for v in self.model.trainable_variables]
        self.path_integral = [np.zeros_like(v.numpy()) for v in self.model.trainable_variables]

    def update_path(self, grads):
        for i, g in enumerate(grads):
            if g is not None:
                delta = self.model.trainable_variables[i].numpy() - self.prev_vars[i]
                self.path_integral[i] += (-g.numpy()) * delta
            self.prev_vars[i] = self.model.trainable_variables[i].numpy().copy()

    def consolidate(self):
        for i, v in enumerate(self.model.trainable_variables):
            delta = v.numpy() - self.prev_vars[i]
            self.w_importance[i] += self.path_integral[i] / (np.square(delta) + self.eps)
            self.path_integral[i] = np.zeros_like(v.numpy())

    def penalty_value(self):
        total = 0.0
        for i, v in enumerate(self.model.trainable_variables):
            total += np.sum(self.w_importance[i] * np.square(v.numpy() - self.prev_vars[i]))
        return self.c * total

# -------------------------------
# Metrics helpers
# -------------------------------
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(10))
    # Macro metrics over seen labels only (avoid penalizing unseen classes)
    seen_labels = np.unique(y_true)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=seen_labels, average="macro", zero_division=0
    )
    return acc, cm, pr, rc, f1

def print_cm(cm):
    print("Confusion matrix (rows=true, cols=pred, labels 09):")
    print(cm)

# -------------------------------
# Training/evaluation runner
# -------------------------------
def run_experiment(mech="naive", rehearsal_size=1000, epochs_per_batch=(2, 2, 3), lrs=(1e-3, 8e-4, 6e-4)):
    model = build_model()
    opt = keras.optimizers.Adam(lrs[0])
    ce_loss = keras.losses.SparseCategoricalCrossentropy()

    buffer = RehearsalBuffer(rehearsal_size) if mech in ["rehearsal", "gen_replay"] else None
    ewc = EWC(model, lam=100.0) if mech == "ewc" else None
    si = SITracker(model, c=1.0) if mech == "si" else None

    results = {
        "acc": [],
        "prec": [],
        "rec": [],
        "f1": [],
        "cm": []
    }

    seen_x, seen_y = [], []

    def train_one_batch(x_b, y_b, epochs, lr):
        nonlocal opt
        opt.learning_rate = lr
        # Merge with buffer (replay/generative replay)
        if buffer is not None:
            x_buf, y_buf = buffer.get()
            if x_buf is not None and len(x_buf) > 0:
                x_b = np.concatenate([x_b, x_buf], axis=0)
                y_b = np.concatenate([y_b, y_buf], axis=0)

        ds = tf.data.Dataset.from_tensor_slices((x_b, y_b)).shuffle(10000).batch(128)
        for _ in range(epochs):
            for xb, yb in ds:
                with tf.GradientTape() as tape:
                    preds = model(xb, training=True)
                    ce = ce_loss(yb, preds)
                    reg = 0.0
                    if ewc is not None:
                        reg += ewc.penalty_value()
                    if si is not None:
                        reg += si.penalty_value()
                    loss = ce + reg
                grads = tape.gradient(loss, model.trainable_variables)
                opt.apply_gradients(zip(grads, model.trainable_variables))
                if si is not None:
                    si.update_path(grads)

        # Update buffers or regularizers post-batch
        if buffer is not None:
            # For "gen_replay" we still use rehearsal buffer as a stub
            buffer.add_batch(x_b, y_b)
        if si is not None:
            si.consolidate()
        if ewc is not None:
            ewc.compute_fisher(x_b, y_b, sample_size=1000)

    # Incremental loop
    for bidx, (x_b, y_b) in enumerate(train_batches):
        print(f"\n=== Mechanism: {mech} | Training on batch {bidx+1} (classes {np.unique(y_b)}) ===")
        train_one_batch(x_b, y_b, epochs_per_batch[bidx], lrs[bidx])

        # Track seen so far
        seen_x.append(x_b)
        seen_y.append(y_b)
        X_seen = np.concatenate(seen_x, axis=0)
        Y_seen = np.concatenate(seen_y, axis=0)

        preds = np.argmax(model.predict(X_seen, verbose=0), axis=1)
        acc, cm, pr, rc, f1 = compute_metrics(Y_seen, preds)
        print(f"Accuracy after batch {bidx+1}: {acc:.4f}")
        print(f"Precision (macro over seen): {pr:.4f} | Recall: {rc:.4f} | F1: {f1:.4f}")
        print_cm(cm)

        results["acc"].append(acc)
        results["prec"].append(pr)
        results["rec"].append(rc)
        results["f1"].append(f1)
        results["cm"].append(cm)

        # Report output-layer changes (static architecture: none added)
        out_layer = model.layers[-1]
        weights, bias = out_layer.get_weights()
        num_outputs = weights.shape[1]
        num_weights = weights.size + bias.size
        print(f"Output layer status: units={num_outputs}, total parameters={num_weights}")
        print("Added this batch: units=0, connections=0 (static 10-way softmax)")

    return results

# -------------------------------
# Run all mechanisms
# -------------------------------
mechanisms = ["naive", "rehearsal", "ewc", "si", "gen_replay"]
results_all = {}

for mech in mechanisms:
    print(f"\nRunning mechanism: {mech}")
    # gen_replay is a rehearsal stub; same settings apply
    res = run_experiment(mech=mech)
    results_all[mech] = res

# -------------------------------
# Consolidated summary table
# -------------------------------
def fmt(x): return f"{x:.3f}"

print("\n=== Consolidated Metrics (over seen classes at each batch) ===")
header = ["Batch"]
for mech in mechanisms:
    header += [f"{mech} acc", f"{mech} prec", f"{mech} rec", f"{mech} f1"]
print("\t".join(header))

for i in range(len(batches)):
    row = [str(i+1)]
    for mech in mechanisms:
        row += [
            fmt(results_all[mech]["acc"][i]),
            fmt(results_all[mech]["prec"][i]),
            fmt(results_all[mech]["rec"][i]),
            fmt(results_all[mech]["f1"][i]),
        ]
    print("\t".join(row))



Running mechanism: naive

=== Mechanism: naive | Training on batch 1 (classes [0 1 2 3 4]) ===
Accuracy after batch 1: 0.9914
Precision (macro over seen): 0.9913 | Recall: 0.9914 | F1: 0.9913
Confusion matrix (rows=true, cols=pred, labels 09):
[[5886    0    4    0   33    0    0    0    0    0]
 [   2 6697    8   11   24    0    0    0    0    0]
 [  16   10 5887    6   39    0    0    0    0    0]
 [  20    8   52 6028   23    0    0    0    0    0]
 [   1    5    1    0 5835    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]]
Output layer status: units=10, total parameters=2570
Added this batch: units=0, connections=0 (static 10-way softmax)

=== Mechanism: naive | Training on batch 2 (classes [5 6 7]) ===
Accuracy after batch 2: 0.3636
Precisio

2025-10-27 13:13:32.013429: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Accuracy after batch 3: 0.1950
Precision (macro over seen): 0.0403 | Recall: 0.1983 | F1: 0.0667
Confusion matrix (rows=true, cols=pred, labels 09):
[[   0    0    0    0    0    0    0    0 3179 2744]
 [   0    0    0    0    0    0    0    0 6243  499]
 [   0    0    0    0    0    0    0    0 5730  228]
 [   0    0    0    0    0    0    0    0 4868 1263]
 [   0    0    0    0    0    0    0    0  348 5494]
 [   0    0    0    0    0    0    0    0 4614  807]
 [   0    0    0    0    0    0    0    0 4139 1779]
 [   0    0    0    0    0    0    0    0  370 5895]
 [   0    0    0    0    0    0    0    0 5815   36]
 [   0    0    0    0    0    0    0    0   64 5885]]
Output layer status: units=10, total parameters=2570
Added this batch: units=0, connections=0 (static 10-way softmax)

Running mechanism: rehearsal

=== Mechanism: rehearsal | Training on batch 1 (classes [0 1 2 3 4]) ===
Accuracy after batch 1: 0.9942
Precision (macro over seen): 0.9941 | Recall: 0.9943 | F1: 0.9942


### Dynamic Neural Architecture
- Batch 1: Digits 0-4
- Batch 2: Digits 5-7
- Batch 3: Digits 8,9

Mechanisms compared:

- Naive fine-tuning
- Rehearsal buffer
- EWC (Elastic Weight Consolidation) with improved regularization
- Synaptic Intelligence (SI) with improved regularization
- Generative replay (stubbed as rehearsal; you can later plug a VAE/GAN)

Key implementation details:
- The output layer expands per batch; hidden layers are kept the same and their weights are copied forward.
- EWC and SI store NumPy snapshots (not graph tensors) to avoid autograph scope issues.
- Evaluation is always on seen so far data; confusion matrix shows labels 0-9 for a consistent layout.

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

# -------------------------------
# Reproducibility and basic config
# -------------------------------
tf.random.set_seed(42)
np.random.seed(42)
H1 = 512
H2 = 256
EPOCHS_PER_BATCH = (3, 3, 4)
LRS = (1e-3, 7e-4, 5e-4)
REHEARSAL_SIZE = 1000
DISTILL_T = 2.0
DISTILL_ALPHA = 0.5
MECHANISMS = ["naive", "rehearsal", "ewc", "si", "gen_replay"]

# -------------------------------
# Load and preprocess MNIST
# -------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

# Class-incremental batches (train and test splits)
batches = [(0, 5), (5, 8), (8, 10)]
train_batches, test_batches = [], []
for start, end in batches:
    mask_tr = np.isin(y_train, np.arange(start, end))
    mask_te = np.isin(y_test, np.arange(start, end))
    train_batches.append((x_train[mask_tr], y_train[mask_tr]))
    test_batches.append((x_test[mask_te], y_test[mask_te]))

# -------------------------------
# Model builders (dynamic head)
# -------------------------------
def build_base(hidden1=H1, hidden2=H2, num_outputs=5):
    inputs = keras.Input(shape=(784,))
    x = layers.Dense(hidden1, activation="relu", name="dense1")(inputs)
    x = layers.Dense(hidden2, activation="relu", name="dense2")(x)
    outputs = layers.Dense(num_outputs, activation="softmax", name="out")(x)
    return keras.Model(inputs, outputs)

def expand_output_layer(prev_model, new_num_outputs):
    # Extract current weights
    d1 = prev_model.get_layer("dense1")
    d2 = prev_model.get_layer("dense2")
    out = prev_model.get_layer("out")
    d1_W, d1_b = d1.get_weights()
    d2_W, d2_b = d2.get_weights()
    out_W, out_b = out.get_weights()

    h1_units = d1_W.shape[1]
    h2_units = d2_W.shape[1]
    old_out_units = out_W.shape[1]

    # Build expanded model
    inputs = keras.Input(shape=(784,))
    x = layers.Dense(h1_units, activation="relu", name="dense1")(inputs)
    x = layers.Dense(h2_units, activation="relu", name="dense2")(x)
    outputs = layers.Dense(new_num_outputs, activation="softmax", name="out")(x)
    new_model = keras.Model(inputs, outputs)

    # Copy hidden weights
    new_model.get_layer("dense1").set_weights([d1_W, d1_b])
    new_model.get_layer("dense2").set_weights([d2_W, d2_b])

    # Expand output weights
    new_W = np.random.normal(scale=0.01, size=(h2_units, new_num_outputs)).astype(np.float32)
    new_b = np.zeros(new_num_outputs, dtype=np.float32)
    new_W[:, :old_out_units] = out_W
    new_b[:old_out_units] = out_b
    new_model.get_layer("out").set_weights([new_W, new_b])

    added_units = new_num_outputs - old_out_units
    added_connections = h2_units * added_units + added_units  # kernel + bias
    return new_model, added_units, added_connections

# -------------------------------
# Rehearsal buffer (reservoir)
# -------------------------------
class RehearsalBuffer:
    def __init__(self, size=REHEARSAL_SIZE):
        self.size = size
        self.data = None
        self.labels = None
    def add_batch(self, x, y):
        if self.data is None:
            cap = min(len(x), self.size)
            self.data = x[:cap].copy()
            self.labels = y[:cap].copy()
        else:
            for i in range(len(x)):
                if len(self.data) < self.size:
                    self.data = np.vstack([self.data, x[i:i+1]])
                    self.labels = np.hstack([self.labels, y[i:i+1]])
                else:
                    j = np.random.randint(0, self.size)
                    self.data[j] = x[i]
                    self.labels[j] = y[i]
    def get(self):
        return self.data, self.labels

# -------------------------------
# EWC and SI (NumPy snapshots)
# -------------------------------
class EWC:
    def __init__(self, model, lam=200.0):
        self.model = model
        self.lam = lam
        self.star_vars = None
        self.fisher = None
    def compute_fisher(self, x, y, sample_size=2000):
        if len(x) == 0: return
        idx = np.random.choice(len(x), min(sample_size, len(x)), replace=False)
        x_s = tf.convert_to_tensor(x[idx]); y_s = tf.convert_to_tensor(y[idx])
        with tf.GradientTape() as tape:
            preds = self.model(x_s, training=False)
            loss = tf.reduce_mean(keras.losses.sparse_categorical_crossentropy(y_s, preds))
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.fisher = [(g.numpy()**2) if g is not None else np.zeros_like(v.numpy())
                       for g, v in zip(grads, self.model.trainable_variables)]
        self.star_vars = [v.numpy().copy() for v in self.model.trainable_variables]
    def penalty_value(self):
        if self.fisher is None or self.star_vars is None: return 0.0
        total = 0.0
        for v, v_star, f in zip(self.model.trainable_variables, self.star_vars, self.fisher):
            total += np.sum(f * np.square(v.numpy() - v_star))
        return (self.lam / 2.0) * total

class SITracker:
    def __init__(self, model, c=2.0, eps=1e-3):
        self.model = model
        self.c = c
        self.eps = eps
        self.prev_vars = [v.numpy().copy() for v in self.model.trainable_variables]
        self.w_importance = [np.zeros_like(v.numpy()) for v in self.model.trainable_variables]
        self.path_integral = [np.zeros_like(v.numpy()) for v in self.model.trainable_variables]
    def update_path(self, grads):
        for i, g in enumerate(grads):
            if g is not None:
                delta = self.model.trainable_variables[i].numpy() - self.prev_vars[i]
                self.path_integral[i] += (-g.numpy()) * delta
            self.prev_vars[i] = self.model.trainable_variables[i].numpy().copy()
    def consolidate(self):
        for i, v in enumerate(self.model.trainable_variables):
            delta = v.numpy() - self.prev_vars[i]
            self.w_importance[i] += self.path_integral[i] / (np.square(delta) + self.eps)
            self.path_integral[i] = np.zeros_like(v.numpy())
    def penalty_value(self):
        total = 0.0
        for i, v in enumerate(self.model.trainable_variables):
            total += np.sum(self.w_importance[i] * np.square(v.numpy() - self.prev_vars[i]))
        return self.c * total

# -------------------------------
# Distillation (Learning without Forgetting)
# -------------------------------
def soft_targets_probs(model, x, temperature=2.0):
    # Softened probabilities from the teacher over its known outputs
    preds = model(x, training=False)  # probs
    preds = tf.clip_by_value(preds, 1e-7, 1.0)
    soft = tf.nn.softmax(tf.math.log(preds) / temperature)
    return soft

def distillation_kl(student_probs, teacher_soft, temperature=2.0):
    # KL(teacher || student) with temperature scaling
    student_probs = tf.clip_by_value(student_probs, 1e-7, 1.0)
    s_soft = tf.nn.softmax(tf.math.log(student_probs) / temperature)
    kl = tf.reduce_mean(tf.reduce_sum(
        teacher_soft * (tf.math.log(teacher_soft + 1e-7) - tf.math.log(s_soft + 1e-7)), axis=1))
    return kl

# -------------------------------
# Metrics helpers
# -------------------------------
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(10))
    seen_labels = np.unique(y_true)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=seen_labels, average="macro", zero_division=0
    )
    return acc, cm, pr, rc, f1

def print_cm(cm):
    print("Confusion matrix (rows=true, cols=pred, labels 09):")
    print(cm)

# -------------------------------
# Training/evaluation runner with dynamic expansion + distillation
# -------------------------------
def run_experiment_dynamic(mech="naive",
                           use_distill_for=("ewc","si"),   # apply LwF to these mechanisms
                           rehearsal_size=REHEARSAL_SIZE,
                           epochs_per_batch=EPOCHS_PER_BATCH,
                           lrs=LRS,
                           hidden1=H1, hidden2=H2,
                           distill_T=DISTILL_T, distill_alpha=DISTILL_ALPHA):
    # Start with 5 outputs (digits 04)
    model = build_base(hidden1=hidden1, hidden2=hidden2, num_outputs=5)
    opt = keras.optimizers.Adam(lrs[0])
    ce_loss = keras.losses.SparseCategoricalCrossentropy()

    buffer = RehearsalBuffer(rehearsal_size) if mech in ["rehearsal", "gen_replay"] else None
    ewc = EWC(model, lam=200.0) if mech == "ewc" else None
    si = SITracker(model, c=2.0) if mech == "si" else None

    results = {"acc": [], "prec": [], "rec": [], "f1": [], "cm": []}
    teacher_model = None  # frozen snapshot before current batch

    def train_one_batch(x_b, y_b, epochs, lr, seen_outputs):
        nonlocal opt
        opt.learning_rate = lr

        # Replay (rehearsal or gen_replay stub)
        if buffer is not None:
            x_buf, y_buf = buffer.get()
            if x_buf is not None and len(x_buf) > 0:
                x_b = np.concatenate([x_b, x_buf], axis=0)
                y_b = np.concatenate([y_b, y_buf], axis=0)

        ds = tf.data.Dataset.from_tensor_slices((x_b, y_b)).shuffle(20000).batch(128)
        for _ in range(epochs):
            for xb, yb in ds:
                with tf.GradientTape() as tape:
                    preds = model(xb, training=True)  # probs over current head
                    ce = ce_loss(yb, preds)
                    reg = 0.0
                    if ewc is not None:
                        reg += ewc.penalty_value()
                    if si is not None:
                        reg += si.penalty_value()
                    loss = ce + reg

                    # Distillation (LwF) for mechanisms opted-in, only on old outputs
                    if (mech in use_distill_for) and (teacher_model is not None):
                        # Teacher knows only old outputs; match student on those indices
                        teacher_soft = soft_targets_probs(teacher_model, xb, temperature=distill_T)
                        student_probs_seen = preds[:, :teacher_soft.shape[1]]
                        dloss = distillation_kl(student_probs_seen, teacher_soft, temperature=distill_T)
                        loss = (1 - distill_alpha) * loss + distill_alpha * dloss

                grads = tape.gradient(loss, model.trainable_variables)
                opt.apply_gradients(zip(grads, model.trainable_variables))
                if si is not None:
                    si.update_path(grads)

        # Post-batch updates
        if buffer is not None:
            buffer.add_batch(x_b, y_b)
        if si is not None:
            si.consolidate()
        if ewc is not None:
            ewc.compute_fisher(x_b, y_b, sample_size=2000)

    # Incremental loop with dynamic expansion of output layer
    for bidx, ((x_b, y_b), (x_te_b, y_te_b)) in enumerate(zip(train_batches, test_batches)):
        # Snapshot teacher before expansion/training (for distillation)
        if mech in use_distill_for:
            teacher_model = keras.models.clone_model(model)
            teacher_model.build((None, 784))
            teacher_model.set_weights(model.get_weights())
            # Teacher stays frozen and is NOT expanded; it only supervises old outputs.

        # Target outputs per batch
        target_outputs = [5, 8, 10][bidx]
        out_W, out_b = model.get_layer("out").get_weights()
        current_outputs = out_W.shape[1]
        added_units = 0
        added_connections = 0
        if current_outputs < target_outputs:
            model, added_units, added_connections = expand_output_layer(model, target_outputs)
            opt = keras.optimizers.Adam(lrs[bidx])
            # Reattach regularizers to new student model
            if isinstance(ewc, EWC):
                ewc = EWC(model, lam=ewc.lam)
            if isinstance(si, SITracker):
                si = SITracker(model, c=si.c)

        print(f"\n=== Mechanism: {mech} | Training on batch {bidx+1} (classes {np.unique(y_b)}) ===")
        print(f"Output layer status before training: units={target_outputs}")
        print(f"Added this batch: units={added_units}, connections={added_connections}")

        # Train current batch
        train_one_batch(x_b, y_b, EPOCHS_PER_BATCH[bidx], LRS[bidx], seen_outputs=target_outputs)

        # Evaluate on cumulative TEST sets up to current batch (04, then 07, then 09)
        cumulative_X = [tb[0] for tb in test_batches[:bidx+1]]
        cumulative_Y = [tb[1] for tb in test_batches[:bidx+1]]
        X_seen_test = np.concatenate(cumulative_X, axis=0)
        Y_seen_test = np.concatenate(cumulative_Y, axis=0)

        preds = np.argmax(model.predict(X_seen_test, verbose=0), axis=1)
        acc, cm, pr, rc, f1 = compute_metrics(Y_seen_test, preds)
        print(f"Accuracy after batch {bidx+1}: {acc:.4f}")
        print(f"Precision (macro over seen): {pr:.4f} | Recall: {rc:.4f} | F1: {f1:.4f}")
        print("Confusion matrix (rows=true, cols=pred, labels 09):")
        print(cm)

        # Final output layer status
        out_W, out_b = model.get_layer("out").get_weights()
        total_params = out_W.size + out_b.size
        print(f"Output layer status after training: units={out_W.shape[1]}, total parameters={total_params}")

        results["acc"].append(acc)
        results["prec"].append(pr)
        results["rec"].append(rc)
        results["f1"].append(f1)
        results["cm"].append(cm)

    return results

# -------------------------------
# Run all mechanisms
# -------------------------------
results_all = {}
for mech in MECHANISMS:
    print(f"\nRunning mechanism: {mech}")
    # gen_replay is a rehearsal stub (same behavior, different label)
    res = run_experiment_dynamic(mech=mech)
    results_all[mech] = res

# -------------------------------
# Consolidated summary table
# -------------------------------
def fmt(x): return f"{x:.3f}"

print("\n=== Consolidated Metrics (cumulative TEST sets at each batch) ===")
header = ["Batch"]
for mech in MECHANISMS:
    header += [f"{mech} acc", f"{mech} prec", f"{mech} rec", f"{mech} f1"]
print("\t".join(header))

for i in range(len(batches)):
    row = [str(i+1)]
    for mech in MECHANISMS:
        row += [
            fmt(results_all[mech]["acc"][i]),
            fmt(results_all[mech]["prec"][i]),
            fmt(results_all[mech]["rec"][i]),
            fmt(results_all[mech]["f1"][i]),
        ]
    print("\t".join(row))



Running mechanism: naive

=== Mechanism: naive | Training on batch 1 (classes [0 1 2 3 4]) ===
Output layer status before training: units=5
Added this batch: units=0, connections=0
Accuracy after batch 1: 0.9914
Precision (macro over seen): 0.9915 | Recall: 0.9913 | F1: 0.9914
Confusion matrix (rows=true, cols=pred, labels 09):
[[ 974    1    3    2    0    0    0    0    0    0]
 [   0 1130    4    1    0    0    0    0    0    0]
 [   4    1 1020    5    2    0    0    0    0    0]
 [   0    0    6 1004    0    0    0    0    0    0]
 [   2    1   10    2  967    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]
 [   0    0    0    0    0    0    0    0    0    0]]
Output layer status after training: units=5, total parameters=1285

=== Mechanism: naive | Training on batch 2 (classes [5 6 7]) ===
Outp

In [11]:
import pandas as pd

# Build a tidy DataFrame from results_all
rows = []
for batch_idx in range(len(batches)):
    row = {"Batch": batch_idx + 1}
    for mech, res in results_all.items():
        row[f"{mech}_acc"]  = res["acc"][batch_idx]
        row[f"{mech}_prec"] = res["prec"][batch_idx]
        row[f"{mech}_rec"]  = res["rec"][batch_idx]
        row[f"{mech}_f1"]   = res["f1"][batch_idx]
    rows.append(row)

df = pd.DataFrame(rows)

# Round for readability
df_rounded = df.round(3)

# Display nicely
print(df_rounded.to_string(index=False))


 Batch  naive_acc  naive_prec  naive_rec  naive_f1  rehearsal_acc  rehearsal_prec  rehearsal_rec  rehearsal_f1  ewc_acc  ewc_prec  ewc_rec  ewc_f1  si_acc  si_prec  si_rec  si_f1  gen_replay_acc  gen_replay_prec  gen_replay_rec  gen_replay_f1
     1      0.991       0.991      0.991     0.991          0.991           0.991          0.991         0.991    0.994     0.994    0.994   0.994   0.995    0.995   0.995  0.995           0.993            0.993           0.993          0.993
     2      0.355       0.142      0.371     0.202          0.924           0.934          0.924         0.922    0.356     0.136    0.372   0.198   0.357    0.259   0.373  0.197           0.927            0.938           0.927          0.926
     3      0.197       0.040      0.199     0.066          0.875           0.912          0.874         0.880    0.252     0.297    0.257   0.162   0.268    0.306   0.273  0.184           0.869            0.909           0.868          0.874


In [14]:
import pandas as pd
from tabulate import tabulate

# Build DataFrame from results_all
rows = []
for batch_idx in range(len(batches)):
    row = {"Batch": batch_idx + 1}
    for mech, res in results_all.items():
        row[f"{mech}_acc"]  = res["acc"][batch_idx]
        row[f"{mech}_prec"] = res["prec"][batch_idx]
        row[f"{mech}_rec"]  = res["rec"][batch_idx]
        row[f"{mech}_f1"]   = res["f1"][batch_idx]
    rows.append(row)

df = pd.DataFrame(rows).round(3)

# Pretty print as a table
print(tabulate(df, headers="keys", tablefmt="grid", showindex=False, floatfmt=".3f"))


+---------+-------------+--------------+-------------+------------+-----------------+------------------+-----------------+----------------+-----------+------------+-----------+----------+----------+-----------+----------+---------+------------------+-------------------+------------------+-----------------+
|   Batch |   naive_acc |   naive_prec |   naive_rec |   naive_f1 |   rehearsal_acc |   rehearsal_prec |   rehearsal_rec |   rehearsal_f1 |   ewc_acc |   ewc_prec |   ewc_rec |   ewc_f1 |   si_acc |   si_prec |   si_rec |   si_f1 |   gen_replay_acc |   gen_replay_prec |   gen_replay_rec |   gen_replay_f1 |
+=========+=============+==============+=============+============+=================+==================+=================+================+===========+============+===========+==========+==========+===========+==========+=========+==================+===================+==================+=================+
|   1.000 |       0.991 |        0.991 |       0.991 |      0.991 |         